In [1]:
import pandas as pd
import sqlite3

In [2]:
# creating database connection
conn = sqlite3.connect('my_database.db')

In [3]:
# checking tables present in the database
query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql_query(query, conn)
print(tables)

               name
0  SuperStoreOrders


In [12]:
for table in tables['name']:
    count = pd.read_sql_query(
        f"SELECT COUNT(*) FROM {table}",
        conn
    ).iloc[0, 0]
    print('-' * 20, f'Table: {table}', '-' * 20)
    print('count of records:', count)
    display(pd.read_sql_query(f"SELECT * FROM {table} LIMIT 5", conn))

-------------------- Table: SuperStoreOrders --------------------
count of records: 51290


,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,...,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year
0,AG-2011-2040,1/1/2011,6/1/2011,Standard Class,Toby Braunhardt,Consumer,Constantine,Algeria,Africa,Africa,...,Office Supplies,Storage,"Tenex Lockers, Blue",408,2,0.0,106.140,35.46,Medium,2011
1,IN-2011-47883,1/1/2011,8/1/2011,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,...,Office Supplies,Supplies,"Acme Trimmer, High Speed",120,3,0.1,36.036,9.72,Medium,2011
2,HU-2011-1220,1/1/2011,5/1/2011,Second Class,Annie Thurman,Consumer,Budapest,Hungary,EMEA,EMEA,...,Office Supplies,Storage,"Tenex Box, Single Width",66,4,0.0,29.640,8.17,High,2011
3,IT-2011-3647632,1/1/2011,5/1/2011,Second Class,Eugene Moren,Home Office,Stockholm,Sweden,EU,North,...,Office Supplies,Paper,"Enermax Note Cards, Premium",45,3,0.5,-26.055,4.82,High,2011
4,IN-2011-47883,1/1/2011,8/1/2011,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,...,Furniture,Furnishings,"Eldon Light Bulb, Duo Pack",114,5,0.1,37.770,4.70,Medium,2011


In [14]:
for table in tables['name']:
    columns = pd.read_sql_query(
        f"PRAGMA table_info({table})",
        conn
    )['name']

    print(f"\nTable: {table}")
    print(columns.tolist())


Table: SuperStoreOrders
['order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_name', 'segment', 'state', 'country', 'market', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit', 'shipping_cost', 'order_priority', 'year']


In [15]:
pd.read_sql_query("SELECT COUNT(*) AS total_rows FROM SuperStoreOrders", conn)

,total_rows
0,51290


In [16]:
pd.read_sql_query("""
SELECT
    MIN(order_date) AS first_order,
    MAX(order_date) AS last_order
FROM SuperStoreOrders;
""", conn)

,first_order,last_order
0,1/1/2011,9/9/2014


# CALCULATING KEY PERFORMANCE INDEX

In [30]:
total_sales= pd.read_sql_query(""" 
SELECT ROUND(SUM(sales),2) AS total_sales
FROM SuperStoreOrders;""", conn)  # how much revenue generated

total_sales

,total_sales
0,7838923.0


In [31]:
total_profit = pd.read_sql_query("""
SELECT ROUND(SUM(profit),2) AS total_profit
FROM SuperStoreOrders;""", conn)  # how much profit generated

total_profit

,total_profit
0,1469034.82


In [32]:
total_orders = pd.read_sql_query("""
SELECT COUNT(DISTINCT order_id) AS total_orders
FROM SuperStoreOrders;""", conn) # how many unique orders placed

total_orders

,total_orders
0,25035


In [33]:
total_customers = pd.read_sql_query("""
SELECT COUNT(DISTINCT customer_name) AS total_customers
FROM SuperStoreOrders;""", conn) # how many unique customers

total_customers

,total_customers
0,795


# CATEGORISING

In [21]:
pd.read_sql_query("""
SELECT category,
       SUM(sales),
       SUM(profit)
FROM SuperStoreOrders
GROUP BY category;""", conn) # sales and profit by category

,category,SUM(sales),SUM(profit)
0,Furniture,2407930.0,286782.25380
1,Office Supplies,2791051.0,518473.83430
2,Technology,2639942.0,663778.73318


In [34]:
## PROFIT MARGIN= (PROFIT/SALES)*100

categorized_KPI= pd.read_sql_query("""
  SELECT
    category,
    ROUND(SUM(sales),2) AS sales,
    ROUND(SUM(profit),2) AS profit,
    ROUND((SUM(profit) * 100.0) / SUM(sales), 2) AS profit_margin
FROM SuperStoreOrders
GROUP BY category;""", conn) # profit margin by category

categorized_KPI

,category,sales,profit,profit_margin
0,Furniture,2407930.0,286782.25,11.91
1,Office Supplies,2791051.0,518473.83,18.58
2,Technology,2639942.0,663778.73,25.14


# SUB-CATEGORY PERFORMANCE

In [24]:
pd.read_sql_query("""
SELECT
    sub_category,
    ROUND(SUM(sales),2) AS sales,
    ROUND(SUM(profit),2) AS profit
FROM SuperStoreOrders
GROUP BY sub_category
ORDER BY profit DESC;
""", conn)

,sub_category,sales,profit
0,Copiers,783282.0,258567.55
1,Phones,871182.0,216717.01
2,Bookcases,812018.0,161924.42
3,Chairs,928424.0,141973.80
4,Appliances,378092.0,141680.59
5,Accessories,570192.0,129626.31
6,Storage,913662.0,108461.49
7,Binders,346228.0,72449.85
8,Paper,244307.0,59207.68
9,Machines,415286.0,58867.87


In [35]:
sub_category_KPI = pd.read_sql_query("""
SELECT
    sub_category,
    ROUND(SUM(sales),2) AS sales,
    ROUND(SUM(profit),2) AS profit,
    ROUND((SUM(profit) * 100.0) / SUM(sales),2) AS profit_margin
FROM SuperStoreOrders
GROUP BY sub_category
ORDER BY profit_margin DESC;
""", conn)

sub_category_KPI

,sub_category,sales,profit,profit_margin
0,Appliances,378092.0,141680.59,37.47
1,Copiers,783282.0,258567.55,33.01
2,Phones,871182.0,216717.01,24.88
3,Paper,244307.0,59207.68,24.23
4,Accessories,570192.0,129626.31,22.73
5,Binders,346228.0,72449.85,20.93
6,Labels,73433.0,15010.51,20.44
7,Bookcases,812018.0,161924.42,19.94
8,Envelopes,170926.0,29601.12,17.32
9,Art,371051.0,57953.91,15.62


In [36]:
## LOSS MAKING SUB-CATEGORIES
loss_making_sub_categories = pd.read_sql_query("""
  SELECT
    sub_category,
    ROUND(SUM(profit),2) AS profit
FROM SuperStoreOrders
GROUP BY sub_category
HAVING SUM(profit) < 0
ORDER BY profit;""", conn)

loss_making_sub_categories

,sub_category,profit
0,Tables,-64083.39


# CUSTOMER ANALYSIS

In [37]:
top_customers_sales = pd.read_sql_query("""
SELECT
    customer_name,
    ROUND(SUM(sales),2) AS sales,
    ROUND(SUM(profit),2) AS profit
FROM SuperStoreOrders
GROUP BY customer_name
ORDER BY sales DESC
LIMIT 10;
""", conn)

top_customers_sales

,customer_name,sales,profit
0,Eric Murdock,19496.0,3306.02
1,John Grady,19485.0,3280.78
2,Maria Etezadi,18244.0,3418.56
3,Theone Pippenger,17421.0,2073.89
4,Dan Reichenbach,16705.0,1984.40
5,Ben Ferrer,16666.0,4135.10
6,Randy Bradley,16403.0,4151.35
7,Mathew Reese,16129.0,3711.23
8,Muhammed Yedwab,16100.0,2642.65
9,Steven Ward,15990.0,2794.73


In [38]:
top_customers_profit= pd.read_sql_query("""
SELECT
    customer_name,
    ROUND(SUM(profit),2) AS profit
FROM SuperStoreOrders
GROUP BY customer_name
ORDER BY profit DESC
LIMIT 10;
""", conn)

top_customers_profit

,customer_name,profit
0,Tamara Chand,8672.90
1,Raymond Buch,8453.05
2,Sanjit Chand,8205.38
3,Hunter Lopez,7816.57
4,Bill Eplett,7410.01
5,Harry Marie,6958.29
6,Susan Pistek,6484.41
7,Mike Gockenbach,6458.68
8,Adrian Barton,6417.28
9,Tom Ashbrook,6311.98


In [39]:
customer_categorize= pd.read_sql_query("""
SELECT
    segment,
    ROUND(SUM(sales),2) AS sales,
    ROUND(SUM(profit),2) AS profit,
    ROUND((SUM(profit)*100.0)/SUM(sales),2) AS profit_margin
FROM SuperStoreOrders
GROUP BY segment
ORDER BY sales DESC;
""", conn)

customer_categorize

,segment,sales,profit,profit_margin
0,Consumer,4060041.0,749239.78,18.45
1,Corporate,2370413.0,442785.86,18.68
2,Home Office,1408469.0,277009.18,19.67


# GEOGRAPHICAL ANALYSIS

In [ ]:
country_df= pd.read_sql_query("""
SELECT 
    country,
    ROUND(SUM(sales),2) AS sales,
    ROUND(SUM(profit),2) AS profit,
    ROUND((SUM(profit)*100.0)/SUM(sales),2) AS profit_margin
FROM SuperStoreOrders
GROUP BY country
ORDER BY sales DESC;
""", conn) # sales, profit and profit margin by country high to low based on sales

country_df


,country,sales,profit,profit_margin
0,United States,1304983.0,286397.02,21.95
1,France,527001.0,109029.00,20.69
2,Australia,516935.0,105484.96,20.41
3,Mexico,416248.0,102818.10,24.70
4,Germany,389504.0,107322.82,27.55
...,...,...,...,...
142,Macedonia,210.0,43.38,20.66
143,Eritrea,188.0,76.20,40.53
144,Armenia,156.0,69.09,44.29
145,Equatorial Guinea,150.0,44.46,29.64


In [42]:
# MARKET ANALYSIS
market_df= pd.read_sql_query("""
SELECT
    market,
    ROUND(SUM(sales),2) AS sales,
    ROUND(SUM(profit),2) AS profit,
    ROUND((SUM(profit)*100.0)/SUM(sales),2) AS profit_margin
FROM SuperStoreOrders
GROUP BY market
ORDER BY sales DESC;
""", conn) # sales, profit and profit margin by market high to low based on sales

market_df

,market,sales,profit,profit_margin
0,APAC,2001990.0,437577.58,21.86
1,EU,1803981.0,372829.74,20.67
2,LATAM,1563597.0,221643.49,14.18
3,US,1304983.0,286397.02,21.95
4,EMEA,575744.0,43897.97,7.62
5,Africa,538302.0,88871.63,16.51
6,Canada,50326.0,17817.39,35.40


In [43]:
# REGIONAL ANALYSIS
region_df = pd.read_sql_query("""
SELECT
    region,
    ROUND(SUM(sales),2) AS sales,
    ROUND(SUM(profit),2) AS profit,
    ROUND((SUM(profit)*100.0)/SUM(sales),2) AS profit_margin
FROM SuperStoreOrders
GROUP BY region
ORDER BY sales DESC;
""", conn) # sales, profit and profit margin by region high to low based on sales

region_df

,region,sales,profit,profit_margin
0,Central,1807431.0,311403.98,17.23
1,South,1031558.0,140355.77,13.61
2,North,790907.0,194597.95,24.60
3,Oceania,625744.0,121666.64,19.44
4,EMEA,575744.0,43897.97,7.62
5,Africa,538302.0,88871.63,16.51
6,Southeast Asia,532448.0,17852.33,3.35
7,North Asia,454002.0,165578.42,36.47
8,West,424418.0,108418.45,25.55
9,Central Asia,389796.0,132480.19,33.99


# DISCOUNT ANALYSIS

TO SHOW HOW THE PROFIT AND SALES CHANGE WITH RESPECT TO HOW MUCH DISCOUNT IS PROVIDED

In [44]:
discount_df = pd.read_sql_query("""
SELECT
    discount,
    ROUND(SUM(sales),2) AS sales,
    ROUND(SUM(profit),2) AS profit
FROM SuperStoreOrders
GROUP BY discount
ORDER BY discount;
""", conn)

discount_df

,discount,sales,profit
0,0.000,4280546.0,1770695.27
1,0.002,174305.0,57976.58
2,0.070,53267.0,21148.50
3,0.100,886353.0,260641.71
4,0.150,162096.0,27375.90
5,0.170,124967.0,28163.07
6,0.200,760886.0,117715.87
7,0.202,15000.0,-595.27
8,0.250,51858.0,800.59
9,0.270,70631.0,-1675.08


In [45]:
discount_margin_df = pd.read_sql_query("""
SELECT
    discount,
    ROUND(SUM(sales),2) AS sales,
    ROUND(SUM(profit),2) AS profit,
    ROUND((SUM(profit)*100.0)/SUM(sales),2) AS profit_margin
FROM SuperStoreOrders
GROUP BY discount
ORDER BY discount;
""", conn)

discount_margin_df

,discount,sales,profit,profit_margin
0,0.000,4280546.0,1770695.27,41.37
1,0.002,174305.0,57976.58,33.26
2,0.070,53267.0,21148.50,39.70
3,0.100,886353.0,260641.71,29.41
4,0.150,162096.0,27375.90,16.89
5,0.170,124967.0,28163.07,22.54
6,0.200,760886.0,117715.87,15.47
7,0.202,15000.0,-595.27,-3.97
8,0.250,51858.0,800.59,1.54
9,0.270,70631.0,-1675.08,-2.37


In [46]:
loss_discount_df = pd.read_sql_query("""
SELECT
    discount,
    ROUND(SUM(sales),2) AS sales,
    ROUND(SUM(profit),2) AS profit
FROM SuperStoreOrders
GROUP BY discount
HAVING SUM(profit) < 0
ORDER BY profit;
""", conn)

loss_discount_df

,discount,sales,profit
0,0.700,112178.0,-186350.48
1,0.600,158432.0,-163954.69
2,0.500,221578.0,-158629.88
3,0.400,425754.0,-143748.46
4,0.800,12499.0,-38616.23
5,0.470,60000.0,-31162.25
6,0.300,105580.0,-19685.85
7,0.350,44199.0,-14169.65
8,0.450,31766.0,-13606.79
9,0.402,32530.0,-11430.45


In [48]:
category_discount_df = pd.read_sql_query("""
SELECT
    category,
    discount,
    ROUND(SUM(sales),2) AS sales,
    ROUND(SUM(profit),2) AS profit
FROM SuperStoreOrders
GROUP BY category, discount
ORDER BY category, discount;
""", conn)

category_discount_df

,category,discount,sales,profit
0,Furniture,0.000,1078730.0,465612.96
1,Furniture,0.070,22816.0,9762.69
2,Furniture,0.100,316994.0,103176.20
3,Furniture,0.150,16103.0,1418.99
4,Furniture,0.200,367605.0,26574.16
5,Furniture,0.250,28309.0,1630.81
6,Furniture,0.270,61897.0,-1078.54
7,Furniture,0.300,104192.0,-20011.89
8,Furniture,0.320,8497.0,-2391.14
9,Furniture,0.350,29211.0,-11237.88


# TIME ANALYSIS

In [49]:
yearly_df = pd.read_sql_query("""
SELECT
    year,
    ROUND(SUM(sales),2) AS sales,
    ROUND(SUM(profit),2) AS profit
FROM SuperStoreOrders
GROUP BY year
ORDER BY year;
""", conn)

yearly_df

,year,sales,profit
0,2011,1378859.0,248940.81
1,2012,1682274.0,307415.28
2,2013,2108044.0,408512.76
3,2014,2669746.0,504165.97


In [51]:
yearly_margin_df = pd.read_sql_query("""
SELECT
    year,
    ROUND(SUM(sales),2) AS sales,
    ROUND(SUM(profit),2) AS profit,
    ROUND((SUM(profit)*100.0)/SUM(sales),2) AS profit_margin
FROM SuperStoreOrders
GROUP BY year
ORDER BY year;
""", conn)

yearly_margin_df

,year,sales,profit,profit_margin
0,2011,1378859.0,248940.81,18.05
1,2012,1682274.0,307415.28,18.27
2,2013,2108044.0,408512.76,19.38
3,2014,2669746.0,504165.97,18.88


In [52]:
year_category_df = pd.read_sql_query("""
SELECT
    year,
    category,
    ROUND(SUM(sales),2) AS sales,
    ROUND(SUM(profit),2) AS profit
FROM SuperStoreOrders
GROUP BY year, category
ORDER BY year;
""", conn)

year_category_df

,year,category,sales,profit
0,2011,Furniture,428117.0,53696.93
1,2011,Office Supplies,492011.0,85996.53
2,2011,Technology,458731.0,109247.35
3,2012,Furniture,496207.0,58132.83
4,2012,Office Supplies,608187.0,103305.51
5,2012,Technology,577880.0,145976.94
6,2013,Furniture,677479.0,85640.43
7,2013,Office Supplies,744624.0,149245.74
8,2013,Technology,685941.0,173626.59
9,2014,Furniture,806127.0,89312.06


# CSV FILES CREATION

In [53]:
categorized_KPI.to_csv('categorized_KPI.csv', index=False)

In [54]:
sub_category_KPI.to_csv('sub_category_KPI.csv', index=False)

In [55]:
customer_categorize.to_csv('customer_categorized_KPI.csv', index=False)

In [ ]:
country_df["geo_type"] = "Country"
market_df["geo_type"] = "Market"
region_df["geo_type"] = "Region"

country_df = country_df.rename(columns={"country": "location"})
market_df = market_df.rename(columns={"market": "location"})
region_df = region_df.rename(columns={"region": "location"})

geography_df = pd.concat(
    [country_df, market_df, region_df],
    ignore_index=True
)

geography_df

,location,sales,profit,profit_margin,geo_type
0,United States,1304983.0,286397.02,21.95,Country
1,France,527001.0,109029.00,20.69,Country
2,Australia,516935.0,105484.96,20.41,Country
3,Mexico,416248.0,102818.10,24.70,Country
4,Germany,389504.0,107322.82,27.55,Country
...,...,...,...,...,...
162,West,424418.0,108418.45,25.55,Region
163,Central Asia,389796.0,132480.19,33.99,Region
164,East,366752.0,91522.78,24.95,Region
165,Caribbean,251495.0,34571.32,13.75,Region


In [57]:
geography_df.to_csv('geography_KPI.csv', index=False)

In [58]:
discount_margin_df.to_csv('discount_margin_KPI.csv', index=False)

In [59]:
yearly_margin_df.to_csv('yearly_margin_KPI.csv', index=False)